In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

# =========================================================
# 0. File paths
# =========================================================
LLAMA_FILE = "greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored.csv"
CHATGPT_FILE = "greenclaims_groundtruth_chatgpt_hybrid_retrieval_fewshot_7metrics_sentence_scored.csv"

OUTPUT_DIR = Path("bt_outputs_rewrite")
OUTPUT_DIR.mkdir(exist_ok=True)

# =========================================================
# 1. Global settings
# =========================================================
RANDOM_SEED = 42
N_PAIRS_TOTAL = 240
PAIR_TIE_MARGIN = 0.5

GREENWASH_LABEL = 1
NON_GREENWASH_LABEL = 0

OPTIONAL_GROUP_COLS = ["Year"]
EXTRA_KEEP_COLS = ["Company", "Year", "sentence", "Type", "Accusation"]

# 互斥分層 quota
PAIR_STRATA_QUOTAS = {
    "cross_label_same_year": 80,
    "cross_label_diff_year": 40,
    "same_label_same_year": 60,
    "same_label_diff_year": 60
}

METRICS = {
    "specificity": {
        "col": "specificity_score",
        "direction": "higher_better"
    },
    "evidence": {
        "col": "evidence_substantiation_score",
        "direction": "higher_better"
    },
    "vagueness": {
        "col": "vagueness_score",
        "direction": "lower_better"
    },
    "commitment": {
        "col": "commitment_score",
        "direction": "higher_better"
    },
    "temporal": {
        "col": "temporal_credibility_score",
        "direction": "higher_better"
    },
    "deflection": {
        "col": "deflection_score",
        "direction": "lower_better"
    },
    "comparability": {
        "col": "comparability_score",
        "direction": "higher_better"
    }
}

# =========================================================
# 2. Utility helpers
# =========================================================
def safe_auc(y_true, y_score):
    y_true = pd.Series(y_true)
    y_score = pd.Series(y_score)
    if y_true.nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


def load_and_validate_csv(file_path):
    df = pd.read_csv(file_path)

    required_cols = ["sen_id", "label"] + [v["col"] for v in METRICS.values()]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError("%s 缺少必要欄位: %s" % (file_path, missing))

    df = df.drop_duplicates(subset=["sen_id"]).reset_index(drop=True)

    keep_cols = list(dict.fromkeys(
        required_cols
        + [c for c in OPTIONAL_GROUP_COLS if c in df.columns]
        + [c for c in EXTRA_KEEP_COLS if c in df.columns]
    ))
    df = df[keep_cols].copy()

    df["sen_id"] = df["sen_id"].astype(str)
    df["label"] = pd.to_numeric(df["label"], errors="coerce")

    for metric_info in METRICS.values():
        col = metric_info["col"]
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "Year" in df.columns:
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

    df = df.dropna(subset=["label"]).reset_index(drop=True)
    df["label"] = df["label"].astype(int)

    return df


# =========================================================
# 3. Pair construction and sampling
# =========================================================
def classify_pair_stratum(a, b):
    same_year = False
    if "Year" in a.index and "Year" in b.index:
        if pd.notna(a["Year"]) and pd.notna(b["Year"]) and a["Year"] == b["Year"]:
            same_year = True

    if a["label"] != b["label"]:
        if same_year:
            return "cross_label_same_year"
        else:
            return "cross_label_diff_year"
    else:
        if same_year:
            return "same_label_same_year"
        else:
            return "same_label_diff_year"


def build_candidate_pairs(df):
    rows = []
    n = len(df)

    for i in range(n):
        for j in range(i + 1, n):
            a = df.iloc[i]
            b = df.iloc[j]

            stratum = classify_pair_stratum(a, b)

            rows.append({
                "sen_id_A": a["sen_id"],
                "sen_id_B": b["sen_id"],
                "label_A": int(a["label"]),
                "label_B": int(b["label"]),
                "pair_stratum": stratum
            })

    out = pd.DataFrame(rows)
    if out.empty:
        raise ValueError("沒有可用的 pair。")
    return out


def sample_fixed_pairs_stratified(df, quotas, n_total=240, random_seed=42):
    rng = np.random.default_rng(random_seed)
    all_pairs = build_candidate_pairs(df)

    sampled_parts = []
    used_idx = pd.Index([])

    def safe_sample(frame, n):
        if len(frame) == 0 or n <= 0:
            return frame.iloc[0:0].copy()
        if len(frame) <= n:
            return frame.copy()
        idx = rng.choice(frame.index, size=n, replace=False)
        return frame.loc[idx].copy()

    # strata 可用數量摘要
    available_counts = all_pairs["pair_stratum"].value_counts(dropna=False).rename_axis("pair_stratum").reset_index(name="available_count")
    available_counts.to_csv(OUTPUT_DIR / "pair_stratum_available_counts.csv", index=False, encoding="utf-8-sig")

    for stratum, quota in quotas.items():
        subset = all_pairs[
            (all_pairs["pair_stratum"] == stratum) &
            (~all_pairs.index.isin(used_idx))
        ].copy()

        s = safe_sample(subset, quota)
        sampled_parts.append(s)
        used_idx = used_idx.union(s.index)

    sampled = pd.concat(sampled_parts, axis=0).drop_duplicates(
        subset=["sen_id_A", "sen_id_B"]
    ).reset_index(drop=True)

    if len(sampled) < n_total:
        remaining = all_pairs[~all_pairs.index.isin(used_idx)].copy()
        need = n_total - len(sampled)
        s_extra = safe_sample(remaining, need)

        sampled = pd.concat([sampled, s_extra], axis=0).drop_duplicates(
            subset=["sen_id_A", "sen_id_B"]
        ).reset_index(drop=True)

    if len(sampled) > n_total:
        idx = rng.choice(sampled.index, size=n_total, replace=False)
        sampled = sampled.loc[idx].reset_index(drop=True)

    return sampled


def compute_pair_sampling_diagnostics(sampled_pairs, df_base):
    # pair-level diagnostics
    pair_diag = sampled_pairs["pair_stratum"].value_counts(dropna=False).rename_axis("pair_stratum").reset_index(name="count")
    pair_diag["ratio"] = pair_diag["count"] / len(sampled_pairs)

    # exposure
    exposure_a = sampled_pairs["sen_id_A"].value_counts()
    exposure_b = sampled_pairs["sen_id_B"].value_counts()
    exposure = exposure_a.add(exposure_b, fill_value=0).rename("n_exposures").reset_index()
    exposure.columns = ["sen_id", "n_exposures"]

    exposure = df_base[["sen_id", "label"]].merge(exposure, on="sen_id", how="left")
    exposure["n_exposures"] = exposure["n_exposures"].fillna(0).astype(int)

    if "Year" in df_base.columns:
        exposure = exposure.merge(df_base[["sen_id", "Year"]], on="sen_id", how="left")

    return pair_diag, exposure


def compute_exposure_summary(exposure_df):
    out = pd.DataFrame([{
        "n_items": len(exposure_df),
        "min_exposure": exposure_df["n_exposures"].min(),
        "max_exposure": exposure_df["n_exposures"].max(),
        "mean_exposure": exposure_df["n_exposures"].mean(),
        "std_exposure": exposure_df["n_exposures"].std(),
        "n_zero_exposure": int((exposure_df["n_exposures"] == 0).sum())
    }])
    return out


# =========================================================
# 4. Pairwise outcome construction
# =========================================================
def compare_pair(score_a, score_b, direction, tie_margin=0.5):
    """
    回傳：
      1 = A wins
      0 = B wins
      None = tie 或缺值
    """
    if pd.isna(score_a) or pd.isna(score_b):
        return None

    diff = score_a - score_b

    if abs(diff) < tie_margin:
        return None

    if direction == "higher_better":
        return 1 if diff > 0 else 0
    elif direction == "lower_better":
        return 1 if diff < 0 else 0
    else:
        raise ValueError("未知 direction: %s" % direction)


def build_metric_pairwise_outcomes(df_scores, sampled_pairs, metric_name, metric_col, direction, tie_margin=0.5):
    score_map = df_scores.set_index("sen_id")[metric_col].to_dict()

    out = sampled_pairs.copy()
    out["metric"] = metric_name
    out["score_A"] = out["sen_id_A"].map(score_map)
    out["score_B"] = out["sen_id_B"].map(score_map)

    out["winner_A"] = [
        compare_pair(a, b, direction, tie_margin=tie_margin)
        for a, b in zip(out["score_A"], out["score_B"])
    ]

    out["is_tie_or_missing"] = out["winner_A"].isna().astype(int)
    return out


# =========================================================
# 5. Bradley-Terry MM
# =========================================================
def fit_bradley_terry_mm(pair_df, item_ids, max_iter=5000, tol=1e-8):
    df = pair_df.dropna(subset=["winner_A"]).copy()
    if df.empty:
        return {sid: 0.0 for sid in item_ids}, {sid: 0.0 for sid in item_ids}

    idx_map = {sid: i for i, sid in enumerate(item_ids)}
    n = len(item_ids)

    wins = np.zeros((n, n), dtype=float)
    comps = np.zeros((n, n), dtype=float)

    for _, row in df.iterrows():
        a = idx_map[row["sen_id_A"]]
        b = idx_map[row["sen_id_B"]]
        winner_a = int(row["winner_A"])

        comps[a, b] += 1
        comps[b, a] += 1

        if winner_a == 1:
            wins[a, b] += 1
        else:
            wins[b, a] += 1

    p = np.ones(n, dtype=float)

    for _ in range(max_iter):
        p_old = p.copy()
        p_new = np.zeros(n, dtype=float)

        for i in range(n):
            w_i = wins[i, :].sum()
            denom = 0.0
            for j in range(n):
                if i == j:
                    continue
                n_ij = comps[i, j]
                if n_ij > 0:
                    denom += n_ij / (p[i] + p[j])

            p_new[i] = w_i / denom if denom > 0 else p[i]

        if np.allclose(p_new, 0):
            p_new = np.ones(n, dtype=float)

        p_new = p_new / np.mean(p_new)

        if np.max(np.abs(p_new - p_old)) < tol:
            p = p_new
            break
        p = p_new

    log_p = np.log(np.clip(p, 1e-12, None))
    bt_scores = {sid: float(log_p[idx_map[sid]]) for sid in item_ids}

    item_comparisons = comps.sum(axis=1)
    comparison_counts = {sid: float(item_comparisons[idx_map[sid]]) for sid in item_ids}

    return bt_scores, comparison_counts


# =========================================================
# 6. Evaluation helpers
# =========================================================
def pairwise_accuracy_against_label(df_scores, sampled_pairs, metric_name, metric_col, direction, positive_label=NON_GREENWASH_LABEL, tie_margin=0.5):
    pair_df = build_metric_pairwise_outcomes(
        df_scores=df_scores,
        sampled_pairs=sampled_pairs,
        metric_name=metric_name,
        metric_col=metric_col,
        direction=direction,
        tie_margin=tie_margin
    )

    pair_df = pair_df.dropna(subset=["winner_A"]).copy()
    pair_df = pair_df[pair_df["label_A"] != pair_df["label_B"]].copy()

    if pair_df.empty:
        return np.nan

    pair_df["gold_winner_A"] = (pair_df["label_A"] == positive_label).astype(int)
    return (pair_df["winner_A"].astype(int) == pair_df["gold_winner_A"].astype(int)).mean()


def summarize_top_bottom_labels(bt_df, bt_col, top_k=10):
    ranked = bt_df.sort_values(bt_col, ascending=False).reset_index(drop=True)
    top = ranked.head(top_k)
    bottom = ranked.tail(top_k)

    return {
        "top_k": top_k,
        "top_label_mean": top["label"].mean(),
        "bottom_label_mean": bottom["label"].mean(),
        "top_greenwash_ratio": (top["label"] == GREENWASH_LABEL).mean(),
        "bottom_greenwash_ratio": (bottom["label"] == GREENWASH_LABEL).mean()
    }


def compute_bt_scores_for_all_metrics(df_scores, sampled_pairs, model_name, tie_margin=0.5):
    item_ids = df_scores["sen_id"].tolist()

    base_cols = ["sen_id", "label"]
    for c in ["Company", "Year", "sentence", "Type", "Accusation"]:
        if c in df_scores.columns:
            base_cols.append(c)

    result = df_scores[base_cols].copy()
    metric_tie_summary_rows = []
    metric_comparison_rows = []

    for metric_name, info in METRICS.items():
        pair_df = build_metric_pairwise_outcomes(
            df_scores=df_scores,
            sampled_pairs=sampled_pairs,
            metric_name=metric_name,
            metric_col=info["col"],
            direction=info["direction"],
            tie_margin=tie_margin
        )

        bt_scores, comparison_counts = fit_bradley_terry_mm(pair_df, item_ids=item_ids)
        result["bt_" + metric_name] = result["sen_id"].map(bt_scores)
        result["n_comparisons_" + metric_name] = result["sen_id"].map(comparison_counts)

        tie_ratio = pair_df["is_tie_or_missing"].mean()
        metric_tie_summary_rows.append({
            "model": model_name,
            "metric": metric_name,
            "n_pairs_total": len(pair_df),
            "n_tie_or_missing": int(pair_df["is_tie_or_missing"].sum()),
            "tie_or_missing_ratio": tie_ratio
        })

        metric_comparison_rows.extend([
            {"model": model_name, "metric": metric_name, "sen_id": sid, "n_comparisons": comparison_counts[sid]}
            for sid in item_ids
        ])

        pair_out_file = OUTPUT_DIR / (model_name + "_pairwise_" + metric_name + ".csv")
        pair_df.to_csv(pair_out_file, index=False, encoding="utf-8-sig")

    tie_summary_df = pd.DataFrame(metric_tie_summary_rows)
    comparison_df = pd.DataFrame(metric_comparison_rows)

    return result, tie_summary_df, comparison_df


def compute_model_comparison_summary(model_name, bt_df, raw_df, sampled_pairs, tie_margin=0.5):
    rows = []

    for metric_name, info in METRICS.items():
        bt_col = "bt_" + metric_name

        auc_greenwash = safe_auc(bt_df["label"], -bt_df[bt_col])

        pair_acc = pairwise_accuracy_against_label(
            df_scores=raw_df,
            sampled_pairs=sampled_pairs,
            metric_name=metric_name,
            metric_col=info["col"],
            direction=info["direction"],
            positive_label=NON_GREENWASH_LABEL,
            tie_margin=tie_margin
        )

        rank_summary = summarize_top_bottom_labels(bt_df, bt_col=bt_col, top_k=10)

        rows.append({
            "model": model_name,
            "metric": metric_name,
            "bt_auc_for_greenwashing": auc_greenwash,
            "pairwise_accuracy_cross_label": pair_acc,
            "top10_greenwash_ratio": rank_summary["top_greenwash_ratio"],
            "bottom10_greenwash_ratio": rank_summary["bottom_greenwash_ratio"]
        })

    return pd.DataFrame(rows)


def compute_spearman_consistency(llama_bt, chatgpt_bt):
    merged = llama_bt[["sen_id"] + ["bt_" + m for m in METRICS.keys()]].merge(
        chatgpt_bt[["sen_id"] + ["bt_" + m for m in METRICS.keys()]],
        on="sen_id",
        suffixes=("_llama", "_chatgpt")
    )

    rows = []
    for metric_name in METRICS.keys():
        col_l = "bt_" + metric_name + "_llama"
        col_c = "bt_" + metric_name + "_chatgpt"

        rho, pval = spearmanr(merged[col_l], merged[col_c], nan_policy="omit")
        rows.append({
            "metric": metric_name,
            "spearman_rho_llama_vs_chatgpt": rho,
            "spearman_pvalue": pval
        })

    return pd.DataFrame(rows)


def export_rankings(bt_df, model_name):
    for metric_name in METRICS.keys():
        bt_col = "bt_" + metric_name

        rank_cols = ["sen_id", "label", bt_col]
        for c in ["Company", "Year", "sentence", "Type", "Accusation"]:
            if c in bt_df.columns:
                rank_cols.append(c)

        # 也一起輸出 comparison 次數
        comparison_col = "n_comparisons_" + metric_name
        if comparison_col in bt_df.columns:
            rank_cols.append(comparison_col)

        rank_df = bt_df[rank_cols].sort_values(bt_col, ascending=False).reset_index(drop=True)
        rank_df["rank"] = np.arange(1, len(rank_df) + 1)

        rank_df.to_csv(
            OUTPUT_DIR / (model_name + "_ranking_" + metric_name + ".csv"),
            index=False,
            encoding="utf-8-sig"
        )


# =========================================================
# 7. Main
# =========================================================
def main():
    if sum(PAIR_STRATA_QUOTAS.values()) != N_PAIRS_TOTAL:
        raise ValueError("PAIR_STRATA_QUOTAS 總和必須等於 N_PAIRS_TOTAL")

    llama = load_and_validate_csv(LLAMA_FILE)
    chatgpt = load_and_validate_csv(CHATGPT_FILE)

    common_ids = sorted(set(llama["sen_id"]).intersection(set(chatgpt["sen_id"])))
    if len(common_ids) == 0:
        raise ValueError("LLaMA 與 ChatGPT 沒有共同的 sen_id")

    llama = llama[llama["sen_id"].isin(common_ids)].copy().reset_index(drop=True)
    chatgpt = chatgpt[chatgpt["sen_id"].isin(common_ids)].copy().reset_index(drop=True)

    label_check = llama[["sen_id", "label"]].merge(
        chatgpt[["sen_id", "label"]],
        on="sen_id",
        suffixes=("_llama", "_chatgpt")
    )
    inconsistent = label_check[label_check["label_llama"] != label_check["label_chatgpt"]]
    if not inconsistent.empty:
        raise ValueError("LLaMA 與 ChatGPT 的 label 不一致，請先檢查資料。")

    sampling_base_cols = ["sen_id", "label"] + [c for c in OPTIONAL_GROUP_COLS if c in llama.columns]
    sampling_base = llama[sampling_base_cols].copy()

    sampled_pairs = sample_fixed_pairs_stratified(
        df=sampling_base,
        quotas=PAIR_STRATA_QUOTAS,
        n_total=N_PAIRS_TOTAL,
        random_seed=RANDOM_SEED
    )
    sampled_pairs.to_csv(OUTPUT_DIR / "fixed_sampled_pairs.csv", index=False, encoding="utf-8-sig")

    pair_diag_df, exposure_df = compute_pair_sampling_diagnostics(sampled_pairs, sampling_base)
    pair_diag_df.to_csv(OUTPUT_DIR / "pair_sampling_diagnostics.csv", index=False, encoding="utf-8-sig")
    exposure_df.to_csv(OUTPUT_DIR / "pair_exposure_diagnostics.csv", index=False, encoding="utf-8-sig")

    exposure_summary_df = compute_exposure_summary(exposure_df)
    exposure_summary_df.to_csv(OUTPUT_DIR / "pair_exposure_summary.csv", index=False, encoding="utf-8-sig")

    print("[INFO] 固定抽樣 pair 數量: %d" % len(sampled_pairs))

    llama_bt, llama_tie_summary, llama_comparison_df = compute_bt_scores_for_all_metrics(
        llama, sampled_pairs, model_name="llama_hybrid", tie_margin=PAIR_TIE_MARGIN
    )
    chatgpt_bt, chatgpt_tie_summary, chatgpt_comparison_df = compute_bt_scores_for_all_metrics(
        chatgpt, sampled_pairs, model_name="chatgpt_hybrid", tie_margin=PAIR_TIE_MARGIN
    )

    llama_bt.to_csv(OUTPUT_DIR / "llama_bt_scores.csv", index=False, encoding="utf-8-sig")
    chatgpt_bt.to_csv(OUTPUT_DIR / "chatgpt_bt_scores.csv", index=False, encoding="utf-8-sig")

    tie_summary_df = pd.concat([llama_tie_summary, chatgpt_tie_summary], axis=0).reset_index(drop=True)
    tie_summary_df.to_csv(OUTPUT_DIR / "tie_summary_by_metric.csv", index=False, encoding="utf-8-sig")

    comparison_summary_df = pd.concat([llama_comparison_df, chatgpt_comparison_df], axis=0).reset_index(drop=True)
    comparison_summary_df.to_csv(OUTPUT_DIR / "comparison_counts_by_metric.csv", index=False, encoding="utf-8-sig")

    llama_summary = compute_model_comparison_summary(
        model_name="LLaMA_Hybrid",
        bt_df=llama_bt,
        raw_df=llama,
        sampled_pairs=sampled_pairs,
        tie_margin=PAIR_TIE_MARGIN
    )
    chatgpt_summary = compute_model_comparison_summary(
        model_name="ChatGPT_Hybrid",
        bt_df=chatgpt_bt,
        raw_df=chatgpt,
        sampled_pairs=sampled_pairs,
        tie_margin=PAIR_TIE_MARGIN
    )

    eval_df = pd.concat([llama_summary, chatgpt_summary], axis=0).reset_index(drop=True)
    eval_df.to_csv(OUTPUT_DIR / "bt_model_comparison_summary.csv", index=False, encoding="utf-8-sig")

    spearman_df = compute_spearman_consistency(llama_bt, chatgpt_bt)
    spearman_df.to_csv(OUTPUT_DIR / "llama_chatgpt_spearman_consistency.csv", index=False, encoding="utf-8-sig")

    export_rankings(llama_bt, "llama_hybrid")
    export_rankings(chatgpt_bt, "chatgpt_hybrid")

    print("[INFO] 完成：")
    print(" - 固定 pairs: %s" % (OUTPUT_DIR / "fixed_sampled_pairs.csv"))
    print(" - Strata available counts: %s" % (OUTPUT_DIR / "pair_stratum_available_counts.csv"))
    print(" - Pair sampling diagnostics: %s" % (OUTPUT_DIR / "pair_sampling_diagnostics.csv"))
    print(" - Pair exposure diagnostics: %s" % (OUTPUT_DIR / "pair_exposure_diagnostics.csv"))
    print(" - Pair exposure summary: %s" % (OUTPUT_DIR / "pair_exposure_summary.csv"))
    print(" - LLaMA BT 分數: %s" % (OUTPUT_DIR / "llama_bt_scores.csv"))
    print(" - ChatGPT BT 分數: %s" % (OUTPUT_DIR / "chatgpt_bt_scores.csv"))
    print(" - Tie summary: %s" % (OUTPUT_DIR / "tie_summary_by_metric.csv"))
    print(" - Comparison counts: %s" % (OUTPUT_DIR / "comparison_counts_by_metric.csv"))
    print(" - 比較摘要: %s" % (OUTPUT_DIR / "bt_model_comparison_summary.csv"))
    print(" - Spearman consistency: %s" % (OUTPUT_DIR / "llama_chatgpt_spearman_consistency.csv"))


if __name__ == "__main__":
    main()

[INFO] 固定抽樣 pair 數量: 240
[INFO] 完成：
 - 固定 pairs: bt_outputs_rewrite\fixed_sampled_pairs.csv
 - Strata available counts: bt_outputs_rewrite\pair_stratum_available_counts.csv
 - Pair sampling diagnostics: bt_outputs_rewrite\pair_sampling_diagnostics.csv
 - Pair exposure diagnostics: bt_outputs_rewrite\pair_exposure_diagnostics.csv
 - Pair exposure summary: bt_outputs_rewrite\pair_exposure_summary.csv
 - LLaMA BT 分數: bt_outputs_rewrite\llama_bt_scores.csv
 - ChatGPT BT 分數: bt_outputs_rewrite\chatgpt_bt_scores.csv
 - Tie summary: bt_outputs_rewrite\tie_summary_by_metric.csv
 - Comparison counts: bt_outputs_rewrite\comparison_counts_by_metric.csv
 - 比較摘要: bt_outputs_rewrite\bt_model_comparison_summary.csv
 - Spearman consistency: bt_outputs_rewrite\llama_chatgpt_spearman_consistency.csv


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

# =========================================================
# 0. File paths
# =========================================================
LLAMA_FILE = "greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored.csv"
CHATGPT_FILE = "greenclaims_groundtruth_chatgpt_hybrid_retrieval_fewshot_7metrics_sentence_scored.csv"

OUTPUT_DIR = Path("bt_outputs_rewrite_2400")
OUTPUT_DIR.mkdir(exist_ok=True)

# =========================================================
# 1. Global settings
# =========================================================
RANDOM_SEED = 42
N_PAIRS_TOTAL = 2400 #最多2278 因為68*67/2=2278
PAIR_TIE_MARGIN = 0.3

GREENWASH_LABEL = 1
NON_GREENWASH_LABEL = 0

OPTIONAL_GROUP_COLS = ["Year"]
EXTRA_KEEP_COLS = ["Company", "Year", "sentence", "Type", "Accusation"]

# 互斥分層 quota：240 -> 2400，整體乘 10
PAIR_STRATA_QUOTAS = {
    "cross_label_same_year": 800,
    "cross_label_diff_year": 400,
    "same_label_same_year": 600,
    "same_label_diff_year": 600
}

METRICS = {
    "specificity": {"col": "specificity_score", "direction": "higher_better"},
    "evidence": {"col": "evidence_substantiation_score", "direction": "higher_better"},
    "vagueness": {"col": "vagueness_score", "direction": "lower_better"},
    "commitment": {"col": "commitment_score", "direction": "higher_better"},
    "temporal": {"col": "temporal_credibility_score", "direction": "higher_better"},
    "deflection": {"col": "deflection_score", "direction": "lower_better"},
    "comparability": {"col": "comparability_score", "direction": "higher_better"}
}

# =========================================================
# 2. Utility helpers
# =========================================================
def safe_auc(y_true, y_score):
    y_true = pd.Series(y_true)
    y_score = pd.Series(y_score)
    if y_true.nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


def load_and_validate_csv(file_path):
    df = pd.read_csv(file_path)

    required_cols = ["sen_id", "label"] + [v["col"] for v in METRICS.values()]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{file_path} 缺少必要欄位: {missing}")

    df = df.drop_duplicates(subset=["sen_id"]).reset_index(drop=True)

    keep_cols = list(dict.fromkeys(
        required_cols
        + [c for c in OPTIONAL_GROUP_COLS if c in df.columns]
        + [c for c in EXTRA_KEEP_COLS if c in df.columns]
    ))
    df = df[keep_cols].copy()

    df["sen_id"] = df["sen_id"].astype(str)
    df["label"] = pd.to_numeric(df["label"], errors="coerce")

    for metric_info in METRICS.values():
        col = metric_info["col"]
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if "Year" in df.columns:
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

    df = df.dropna(subset=["label"]).reset_index(drop=True)
    df["label"] = df["label"].astype(int)

    return df


# =========================================================
# 3. Pair construction and sampling
# =========================================================
def classify_pair_stratum(a, b):
    same_year = False
    if "Year" in a.index and "Year" in b.index:
        if pd.notna(a["Year"]) and pd.notna(b["Year"]) and a["Year"] == b["Year"]:
            same_year = True

    if a["label"] != b["label"]:
        return "cross_label_same_year" if same_year else "cross_label_diff_year"
    return "same_label_same_year" if same_year else "same_label_diff_year"


def build_candidate_pairs(df):
    rows = []
    n = len(df)

    for i in range(n):
        for j in range(i + 1, n):
            a = df.iloc[i]
            b = df.iloc[j]
            rows.append({
                "sen_id_A": a["sen_id"],
                "sen_id_B": b["sen_id"],
                "label_A": int(a["label"]),
                "label_B": int(b["label"]),
                "pair_stratum": classify_pair_stratum(a, b)
            })

    out = pd.DataFrame(rows)
    if out.empty:
        raise ValueError("沒有可用的 pair。")
    return out


def sample_fixed_pairs_stratified(df, quotas, n_total=2400, random_seed=42):
    rng = np.random.default_rng(random_seed)
    all_pairs = build_candidate_pairs(df)

    sampled_parts = []
    used_idx = pd.Index([])

    def safe_sample(frame, n):
        if len(frame) == 0 or n <= 0:
            return frame.iloc[0:0].copy()
        if len(frame) <= n:
            return frame.copy()
        idx = rng.choice(frame.index, size=n, replace=False)
        return frame.loc[idx].copy()

    available_counts = (
        all_pairs["pair_stratum"]
        .value_counts(dropna=False)
        .rename_axis("pair_stratum")
        .reset_index(name="available_count")
    )
    available_counts.to_csv(OUTPUT_DIR / "pair_stratum_available_counts.csv", index=False, encoding="utf-8-sig")

    for stratum, quota in quotas.items():
        subset = all_pairs[
            (all_pairs["pair_stratum"] == stratum) &
            (~all_pairs.index.isin(used_idx))
        ].copy()

        s = safe_sample(subset, quota)
        sampled_parts.append(s)
        used_idx = used_idx.union(s.index)

    sampled = pd.concat(sampled_parts, axis=0).drop_duplicates(
        subset=["sen_id_A", "sen_id_B"]
    ).reset_index(drop=True)

    if len(sampled) < n_total:
        remaining = all_pairs[~all_pairs.index.isin(used_idx)].copy()
        need = n_total - len(sampled)
        s_extra = safe_sample(remaining, need)
        sampled = pd.concat([sampled, s_extra], axis=0).drop_duplicates(
            subset=["sen_id_A", "sen_id_B"]
        ).reset_index(drop=True)

    if len(sampled) > n_total:
        idx = rng.choice(sampled.index, size=n_total, replace=False)
        sampled = sampled.loc[idx].reset_index(drop=True)

    return sampled


def compute_pair_sampling_diagnostics(sampled_pairs, df_base):
    pair_diag = sampled_pairs["pair_stratum"].value_counts(dropna=False).rename_axis(
        "pair_stratum"
    ).reset_index(name="count")
    pair_diag["ratio"] = pair_diag["count"] / len(sampled_pairs)

    exposure_a = sampled_pairs["sen_id_A"].value_counts()
    exposure_b = sampled_pairs["sen_id_B"].value_counts()
    exposure = exposure_a.add(exposure_b, fill_value=0).rename("n_exposures").reset_index()
    exposure.columns = ["sen_id", "n_exposures"]

    exposure = df_base[["sen_id", "label"]].merge(exposure, on="sen_id", how="left")
    exposure["n_exposures"] = exposure["n_exposures"].fillna(0).astype(int)

    if "Year" in df_base.columns:
        exposure = exposure.merge(df_base[["sen_id", "Year"]], on="sen_id", how="left")

    return pair_diag, exposure


def compute_exposure_summary(exposure_df):
    return pd.DataFrame([{
        "n_items": len(exposure_df),
        "min_exposure": exposure_df["n_exposures"].min(),
        "max_exposure": exposure_df["n_exposures"].max(),
        "mean_exposure": exposure_df["n_exposures"].mean(),
        "std_exposure": exposure_df["n_exposures"].std(),
        "n_zero_exposure": int((exposure_df["n_exposures"] == 0).sum())
    }])


# =========================================================
# 4. Pairwise outcome construction
# =========================================================
def compare_pair(score_a, score_b, direction, tie_margin=0.3):
    if pd.isna(score_a) or pd.isna(score_b):
        return None

    diff = score_a - score_b

    if abs(diff) < tie_margin:
        return None

    if direction == "higher_better":
        return 1 if diff > 0 else 0
    if direction == "lower_better":
        return 1 if diff < 0 else 0
    raise ValueError(f"未知 direction: {direction}")


def build_metric_pairwise_outcomes(df_scores, sampled_pairs, metric_name, metric_col, direction, tie_margin=0.3):
    score_map = df_scores.set_index("sen_id")[metric_col].to_dict()

    out = sampled_pairs.copy()
    out["metric"] = metric_name
    out["score_A"] = out["sen_id_A"].map(score_map)
    out["score_B"] = out["sen_id_B"].map(score_map)

    out["winner_A"] = [
        compare_pair(a, b, direction, tie_margin=tie_margin)
        for a, b in zip(out["score_A"], out["score_B"])
    ]

    out["is_tie_or_missing"] = out["winner_A"].isna().astype(int)
    return out


# =========================================================
# 5. Bradley-Terry MM
# =========================================================
def fit_bradley_terry_mm(pair_df, item_ids, max_iter=5000, tol=1e-8):
    df = pair_df.dropna(subset=["winner_A"]).copy()
    if df.empty:
        return {sid: 0.0 for sid in item_ids}, {sid: 0.0 for sid in item_ids}

    idx_map = {sid: i for i, sid in enumerate(item_ids)}
    n = len(item_ids)

    wins = np.zeros((n, n), dtype=float)
    comps = np.zeros((n, n), dtype=float)

    for _, row in df.iterrows():
        a = idx_map[row["sen_id_A"]]
        b = idx_map[row["sen_id_B"]]
        winner_a = int(row["winner_A"])

        comps[a, b] += 1
        comps[b, a] += 1

        if winner_a == 1:
            wins[a, b] += 1
        else:
            wins[b, a] += 1

    p = np.ones(n, dtype=float)

    for _ in range(max_iter):
        p_old = p.copy()
        p_new = np.zeros(n, dtype=float)

        for i in range(n):
            w_i = wins[i, :].sum()
            denom = 0.0
            for j in range(n):
                if i == j:
                    continue
                n_ij = comps[i, j]
                if n_ij > 0:
                    denom += n_ij / (p[i] + p[j])

            p_new[i] = w_i / denom if denom > 0 else p[i]

        if np.allclose(p_new, 0):
            p_new = np.ones(n, dtype=float)

        p_new = p_new / np.mean(p_new)

        if np.max(np.abs(p_new - p_old)) < tol:
            p = p_new
            break
        p = p_new

    log_p = np.log(np.clip(p, 1e-12, None))
    bt_scores = {sid: float(log_p[idx_map[sid]]) for sid in item_ids}

    item_comparisons = comps.sum(axis=1)
    comparison_counts = {sid: float(item_comparisons[idx_map[sid]]) for sid in item_ids}

    return bt_scores, comparison_counts


# =========================================================
# 6. Evaluation helpers
# =========================================================
def pairwise_accuracy_against_label(df_scores, sampled_pairs, metric_name, metric_col, direction,
                                    positive_label=NON_GREENWASH_LABEL, tie_margin=0.3):
    pair_df = build_metric_pairwise_outcomes(
        df_scores=df_scores,
        sampled_pairs=sampled_pairs,
        metric_name=metric_name,
        metric_col=metric_col,
        direction=direction,
        tie_margin=tie_margin
    )

    pair_df = pair_df.dropna(subset=["winner_A"]).copy()
    pair_df = pair_df[pair_df["label_A"] != pair_df["label_B"]].copy()

    if pair_df.empty:
        return np.nan

    pair_df["gold_winner_A"] = (pair_df["label_A"] == positive_label).astype(int)
    return (pair_df["winner_A"].astype(int) == pair_df["gold_winner_A"].astype(int)).mean()


def summarize_top_bottom_labels(bt_df, bt_col, top_k=10):
    ranked = bt_df.sort_values(bt_col, ascending=False).reset_index(drop=True)
    top = ranked.head(top_k)
    bottom = ranked.tail(top_k)

    return {
        "top_k": top_k,
        "top_greenwash_ratio": (top["label"] == GREENWASH_LABEL).mean(),
        "bottom_greenwash_ratio": (bottom["label"] == GREENWASH_LABEL).mean()
    }


def compute_bt_scores_for_all_metrics(df_scores, sampled_pairs, model_name, tie_margin=0.3):
    item_ids = df_scores["sen_id"].tolist()

    base_cols = ["sen_id", "label"]
    for c in ["Company", "Year", "sentence", "Type", "Accusation"]:
        if c in df_scores.columns:
            base_cols.append(c)

    result = df_scores[base_cols].copy()
    metric_tie_summary_rows = []
    metric_comparison_rows = []

    for metric_name, info in METRICS.items():
        pair_df = build_metric_pairwise_outcomes(
            df_scores=df_scores,
            sampled_pairs=sampled_pairs,
            metric_name=metric_name,
            metric_col=info["col"],
            direction=info["direction"],
            tie_margin=tie_margin
        )

        bt_scores, comparison_counts = fit_bradley_terry_mm(pair_df, item_ids=item_ids)
        result["bt_" + metric_name] = result["sen_id"].map(bt_scores)
        result["n_comparisons_" + metric_name] = result["sen_id"].map(comparison_counts)

        tie_ratio = pair_df["is_tie_or_missing"].mean()
        metric_tie_summary_rows.append({
            "model": model_name,
            "metric": metric_name,
            "n_pairs_total": len(pair_df),
            "n_tie_or_missing": int(pair_df["is_tie_or_missing"].sum()),
            "tie_or_missing_ratio": tie_ratio
        })

        metric_comparison_rows.extend([
            {"model": model_name, "metric": metric_name, "sen_id": sid, "n_comparisons": comparison_counts[sid]}
            for sid in item_ids
        ])

        pair_df.to_csv(OUTPUT_DIR / f"{model_name}_pairwise_{metric_name}.csv", index=False, encoding="utf-8-sig")

    return pd.DataFrame(result), pd.DataFrame(metric_tie_summary_rows), pd.DataFrame(metric_comparison_rows)


def compute_model_comparison_summary(model_name, bt_df, raw_df, sampled_pairs, tie_margin=0.3):
    rows = []

    for metric_name, info in METRICS.items():
        bt_col = "bt_" + metric_name

        auc_greenwash = safe_auc(bt_df["label"], -bt_df[bt_col])

        pair_acc = pairwise_accuracy_against_label(
            df_scores=raw_df,
            sampled_pairs=sampled_pairs,
            metric_name=metric_name,
            metric_col=info["col"],
            direction=info["direction"],
            positive_label=NON_GREENWASH_LABEL,
            tie_margin=tie_margin
        )

        rank_summary = summarize_top_bottom_labels(bt_df, bt_col=bt_col, top_k=10)

        rows.append({
            "model": model_name,
            "metric": metric_name,
            "bt_auc_for_greenwashing": auc_greenwash,
            "pairwise_accuracy_cross_label": pair_acc,
            "top10_greenwash_ratio": rank_summary["top_greenwash_ratio"],
            "bottom10_greenwash_ratio": rank_summary["bottom_greenwash_ratio"]
        })

    return pd.DataFrame(rows)


def compute_spearman_consistency(llama_bt, chatgpt_bt):
    merged = llama_bt[["sen_id"] + ["bt_" + m for m in METRICS.keys()]].merge(
        chatgpt_bt[["sen_id"] + ["bt_" + m for m in METRICS.keys()]],
        on="sen_id",
        suffixes=("_llama", "_chatgpt")
    )

    rows = []
    for metric_name in METRICS.keys():
        col_l = "bt_" + metric_name + "_llama"
        col_c = "bt_" + metric_name + "_chatgpt"
        rho, pval = spearmanr(merged[col_l], merged[col_c], nan_policy="omit")
        rows.append({
            "metric": metric_name,
            "spearman_rho_llama_vs_chatgpt": rho,
            "spearman_pvalue": pval
        })

    return pd.DataFrame(rows)


def export_rankings(bt_df, model_name):
    for metric_name in METRICS.keys():
        bt_col = "bt_" + metric_name
        rank_cols = ["sen_id", "label", bt_col]

        for c in ["Company", "Year", "sentence", "Type", "Accusation"]:
            if c in bt_df.columns:
                rank_cols.append(c)

        comparison_col = "n_comparisons_" + metric_name
        if comparison_col in bt_df.columns:
            rank_cols.append(comparison_col)

        rank_df = bt_df[rank_cols].sort_values(bt_col, ascending=False).reset_index(drop=True)
        rank_df["rank"] = np.arange(1, len(rank_df) + 1)

        rank_df.to_csv(
            OUTPUT_DIR / f"{model_name}_ranking_{metric_name}.csv",
            index=False,
            encoding="utf-8-sig"
        )


# =========================================================
# 7. Main
# =========================================================
def main():
    if sum(PAIR_STRATA_QUOTAS.values()) != N_PAIRS_TOTAL:
        raise ValueError("PAIR_STRATA_QUOTAS 總和必須等於 N_PAIRS_TOTAL")

    llama = load_and_validate_csv(LLAMA_FILE)
    chatgpt = load_and_validate_csv(CHATGPT_FILE)

    common_ids = sorted(set(llama["sen_id"]).intersection(set(chatgpt["sen_id"])))
    if len(common_ids) == 0:
        raise ValueError("LLaMA 與 ChatGPT 沒有共同的 sen_id")

    llama = llama[llama["sen_id"].isin(common_ids)].copy().reset_index(drop=True)
    chatgpt = chatgpt[chatgpt["sen_id"].isin(common_ids)].copy().reset_index(drop=True)

    label_check = llama[["sen_id", "label"]].merge(
        chatgpt[["sen_id", "label"]],
        on="sen_id",
        suffixes=("_llama", "_chatgpt")
    )
    inconsistent = label_check[label_check["label_llama"] != label_check["label_chatgpt"]]
    if not inconsistent.empty:
        raise ValueError("LLaMA 與 ChatGPT 的 label 不一致，請先檢查資料。")

    sampling_base_cols = ["sen_id", "label"] + [c for c in OPTIONAL_GROUP_COLS if c in llama.columns]
    sampling_base = llama[sampling_base_cols].copy()

    sampled_pairs = sample_fixed_pairs_stratified(
        df=sampling_base,
        quotas=PAIR_STRATA_QUOTAS,
        n_total=N_PAIRS_TOTAL,
        random_seed=RANDOM_SEED
    )
    sampled_pairs.to_csv(OUTPUT_DIR / "fixed_sampled_pairs.csv", index=False, encoding="utf-8-sig")

    pair_diag_df, exposure_df = compute_pair_sampling_diagnostics(sampled_pairs, sampling_base)
    pair_diag_df.to_csv(OUTPUT_DIR / "pair_sampling_diagnostics.csv", index=False, encoding="utf-8-sig")
    exposure_df.to_csv(OUTPUT_DIR / "pair_exposure_diagnostics.csv", index=False, encoding="utf-8-sig")

    exposure_summary_df = compute_exposure_summary(exposure_df)
    exposure_summary_df.to_csv(OUTPUT_DIR / "pair_exposure_summary.csv", index=False, encoding="utf-8-sig")

    print(f"[INFO] 固定抽樣 pair 數量: {len(sampled_pairs)}")

    llama_bt, llama_tie_summary, llama_comparison_df = compute_bt_scores_for_all_metrics(
        llama, sampled_pairs, model_name="llama_hybrid", tie_margin=PAIR_TIE_MARGIN
    )
    chatgpt_bt, chatgpt_tie_summary, chatgpt_comparison_df = compute_bt_scores_for_all_metrics(
        chatgpt, sampled_pairs, model_name="chatgpt_hybrid", tie_margin=PAIR_TIE_MARGIN
    )

    llama_bt.to_csv(OUTPUT_DIR / "llama_bt_scores.csv", index=False, encoding="utf-8-sig")
    chatgpt_bt.to_csv(OUTPUT_DIR / "chatgpt_bt_scores.csv", index=False, encoding="utf-8-sig")

    tie_summary_df = pd.concat([llama_tie_summary, chatgpt_tie_summary], axis=0).reset_index(drop=True)
    tie_summary_df.to_csv(OUTPUT_DIR / "tie_summary_by_metric.csv", index=False, encoding="utf-8-sig")

    comparison_summary_df = pd.concat([llama_comparison_df, chatgpt_comparison_df], axis=0).reset_index(drop=True)
    comparison_summary_df.to_csv(OUTPUT_DIR / "comparison_counts_by_metric.csv", index=False, encoding="utf-8-sig")

    llama_summary = compute_model_comparison_summary(
        model_name="LLaMA_Hybrid",
        bt_df=llama_bt,
        raw_df=llama,
        sampled_pairs=sampled_pairs,
        tie_margin=PAIR_TIE_MARGIN
    )
    chatgpt_summary = compute_model_comparison_summary(
        model_name="ChatGPT_Hybrid",
        bt_df=chatgpt_bt,
        raw_df=chatgpt,
        sampled_pairs=sampled_pairs,
        tie_margin=PAIR_TIE_MARGIN
    )

    eval_df = pd.concat([llama_summary, chatgpt_summary], axis=0).reset_index(drop=True)
    eval_df.to_csv(OUTPUT_DIR / "bt_model_comparison_summary.csv", index=False, encoding="utf-8-sig")

    spearman_df = compute_spearman_consistency(llama_bt, chatgpt_bt)
    spearman_df.to_csv(OUTPUT_DIR / "llama_chatgpt_spearman_consistency.csv", index=False, encoding="utf-8-sig")

    export_rankings(llama_bt, "llama_hybrid")
    export_rankings(chatgpt_bt, "chatgpt_hybrid")

    print("[INFO] 完成")
    print(" - 固定 pairs:", OUTPUT_DIR / "fixed_sampled_pairs.csv")
    print(" - Pair sampling diagnostics:", OUTPUT_DIR / "pair_sampling_diagnostics.csv")
    print(" - Pair exposure summary:", OUTPUT_DIR / "pair_exposure_summary.csv")
    print(" - Tie summary:", OUTPUT_DIR / "tie_summary_by_metric.csv")
    print(" - 比較摘要:", OUTPUT_DIR / "bt_model_comparison_summary.csv")


if __name__ == "__main__":
    main()

[INFO] 固定抽樣 pair 數量: 2278
[INFO] 完成
 - 固定 pairs: bt_outputs_rewrite_2400\fixed_sampled_pairs.csv
 - Pair sampling diagnostics: bt_outputs_rewrite_2400\pair_sampling_diagnostics.csv
 - Pair exposure summary: bt_outputs_rewrite_2400\pair_exposure_summary.csv
 - Tie summary: bt_outputs_rewrite_2400\tie_summary_by_metric.csv
 - 比較摘要: bt_outputs_rewrite_2400\bt_model_comparison_summary.csv


In [ ]:
#算一致性 pairwise_agreement
import pandas as pd
import numpy as np

PAIR_TIE_MARGIN = 0.3

METRICS = {
    "specificity": {"col": "specificity_score", "direction": "higher_better"},
    "evidence": {"col": "evidence_substantiation_score", "direction": "higher_better"},
    "vagueness": {"col": "vagueness_score", "direction": "lower_better"},
    "commitment": {"col": "commitment_score", "direction": "higher_better"},
    "temporal": {"col": "temporal_credibility_score", "direction": "higher_better"},
    "deflection": {"col": "deflection_score", "direction": "lower_better"},
    "comparability": {"col": "comparability_score", "direction": "higher_better"}
}

def compare_pair(score_a, score_b, direction, tie_margin=0.3):
    if pd.isna(score_a) or pd.isna(score_b):
        return np.nan

    diff = score_a - score_b

    if abs(diff) < tie_margin:
        return np.nan

    if direction == "higher_better":
        return 1 if diff > 0 else 0
    elif direction == "lower_better":
        return 1 if diff < 0 else 0
    else:
        raise ValueError(f"Unknown direction: {direction}")

def build_metric_pairwise_outcomes(df_scores, sampled_pairs, metric_col, direction, tie_margin=0.3):
    score_map = df_scores.set_index("sen_id")[metric_col].to_dict()

    out = sampled_pairs.copy()
    out["score_A"] = out["sen_id_A"].map(score_map)
    out["score_B"] = out["sen_id_B"].map(score_map)

    out["winner_A"] = [
        compare_pair(a, b, direction, tie_margin)
        for a, b in zip(out["score_A"], out["score_B"])
    ]

    return out

def compute_pairwise_agreement(llama_df, chatgpt_df, sampled_pairs, metric_name, metric_col, direction, tie_margin=0.3):
    llama_pairs = build_metric_pairwise_outcomes(
        df_scores=llama_df,
        sampled_pairs=sampled_pairs,
        metric_col=metric_col,
        direction=direction,
        tie_margin=tie_margin
    )

    chatgpt_pairs = build_metric_pairwise_outcomes(
        df_scores=chatgpt_df,
        sampled_pairs=sampled_pairs,
        metric_col=metric_col,
        direction=direction,
        tie_margin=tie_margin
    )

    merged = llama_pairs[["sen_id_A", "sen_id_B", "winner_A"]].merge(
        chatgpt_pairs[["sen_id_A", "sen_id_B", "winner_A"]],
        on=["sen_id_A", "sen_id_B"],
        suffixes=("_llama", "_chatgpt")
    )

    # 只保留兩模型都有明確 winner 的 pair
    merged = merged.dropna(subset=["winner_A_llama", "winner_A_chatgpt"]).copy()

    if len(merged) == 0:
        return {
            "metric": metric_name,
            "pairwise_agreement": np.nan,
            "n_valid_pairs": 0
        }

    agreement = (merged["winner_A_llama"] == merged["winner_A_chatgpt"]).mean()

    return {
        "metric": metric_name,
        "pairwise_agreement": agreement,
        "n_valid_pairs": len(merged)
    }

# 使用方式
llama_df = pd.read_csv("greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored.csv")
chatgpt_df = pd.read_csv("greenclaims_groundtruth_chatgpt_hybrid_retrieval_fewshot_7metrics_sentence_scored.csv")
sampled_pairs = pd.read_csv("bt_outputs_rewrite_2400/fixed_sampled_pairs.csv")

rows = []
for metric_name, info in METRICS.items():
    result = compute_pairwise_agreement(
        llama_df=llama_df,
        chatgpt_df=chatgpt_df,
        sampled_pairs=sampled_pairs,
        metric_name=metric_name,
        metric_col=info["col"],
        direction=info["direction"],
        tie_margin=PAIR_TIE_MARGIN
    )
    rows.append(result)

agreement_df = pd.DataFrame(rows)
print(agreement_df)
agreement_df.to_csv("bt_outputs_rewrite_2400/cross_model_pairwise_agreement.csv", index=False, encoding="utf-8-sig")

          metric  pairwise_agreement  n_valid_pairs
0    specificity            0.861481           1350
1       evidence            0.918340            747
2      vagueness            0.791916           1336
3     commitment            0.895147           1154
4       temporal            0.950089            561
5     deflection            0.605263            912
6  comparability            0.741047            726


In [4]:
# 直接算 Spearman 相關性
import pandas as pd
from scipy.stats import spearmanr

# 讀檔
llama_bt = pd.read_csv("bt_outputs_rewrite_2400/llama_bt_scores.csv")
chatgpt_bt = pd.read_csv("bt_outputs_rewrite_2400/chatgpt_bt_scores.csv")

# 你的 metrics（要跟欄位一致）
metrics = [
    "specificity",
    "evidence",
    "vagueness",
    "commitment",
    "temporal",
    "deflection",
    "comparability"
]

# merge
merged = llama_bt[["sen_id"] + ["bt_" + m for m in metrics]].merge(
    chatgpt_bt[["sen_id"] + ["bt_" + m for m in metrics]],
    on="sen_id",
    suffixes=("_llama", "_chatgpt")
)

# 算 Spearman
rows = []

for m in metrics:
    col_l = "bt_" + m + "_llama"
    col_c = "bt_" + m + "_chatgpt"

    rho, pval = spearmanr(merged[col_l], merged[col_c])

    rows.append({
        "metric": m,
        "spearman_rho": rho,
        "p_value": pval
    })

spearman_df = pd.DataFrame(rows)

print(spearman_df)

# 存檔（這步很重要）
spearman_df.to_csv(
    "bt_outputs_rewrite_2400/cross_model_spearman.csv",
    index=False,
    encoding="utf-8-sig"
)

          metric  spearman_rho       p_value
0    specificity      0.650825  1.876289e-09
1       evidence      0.552413  1.039783e-06
2      vagueness      0.526553  3.982874e-06
3     commitment      0.641289  3.823329e-09
4       temporal      0.557170  8.021061e-07
5     deflection      0.159281  1.944900e-01
6  comparability      0.297813  1.364233e-02


In [5]:
spearman_df = pd.read_csv("bt_outputs_rewrite_2400/cross_model_spearman.csv")
agreement_df = pd.read_csv("bt_outputs_rewrite_2400/cross_model_pairwise_agreement.csv")

final_consistency_df = spearman_df.merge(agreement_df, on="metric", how="outer")
print(final_consistency_df)

final_consistency_df.to_csv(
    "bt_outputs_rewrite_2400/cross_model_consistency_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

          metric  spearman_rho       p_value  pairwise_agreement  \
0    specificity      0.650825  1.876289e-09            0.861481   
1       evidence      0.552413  1.039783e-06            0.918340   
2      vagueness      0.526553  3.982874e-06            0.791916   
3     commitment      0.641289  3.823329e-09            0.895147   
4       temporal      0.557170  8.021061e-07            0.950089   
5     deflection      0.159281  1.944900e-01            0.605263   
6  comparability      0.297813  1.364233e-02            0.741047   

   n_valid_pairs  
0           1350  
1            747  
2           1336  
3           1154  
4            561  
5            912  
6            726  


In [6]:
#modelbattle
import pandas as pd
import numpy as np

PAIR_TIE_MARGIN = 0.3
NON_GREENWASH_LABEL = 0
GREENWASH_LABEL = 1

METRICS = {
    "specificity": {"col": "specificity_score", "direction": "higher_better"},
    "evidence": {"col": "evidence_substantiation_score", "direction": "higher_better"},
    "vagueness": {"col": "vagueness_score", "direction": "lower_better"},
    "commitment": {"col": "commitment_score", "direction": "higher_better"},
    "temporal": {"col": "temporal_credibility_score", "direction": "higher_better"},
    "deflection": {"col": "deflection_score", "direction": "lower_better"},
    "comparability": {"col": "comparability_score", "direction": "higher_better"}
}

def compare_pair(score_a, score_b, direction, tie_margin=0.3):
    if pd.isna(score_a) or pd.isna(score_b):
        return np.nan

    diff = score_a - score_b

    if abs(diff) < tie_margin:
        return np.nan

    if direction == "higher_better":
        return 1 if diff > 0 else 0   # 1 = A wins, 0 = B wins
    elif direction == "lower_better":
        return 1 if diff < 0 else 0
    else:
        raise ValueError(f"Unknown direction: {direction}")

def build_metric_pairwise_outcomes(df_scores, sampled_pairs, metric_col, direction, tie_margin=0.3):
    score_map = df_scores.set_index("sen_id")[metric_col].to_dict()

    out = sampled_pairs.copy()
    out["score_A"] = out["sen_id_A"].map(score_map)
    out["score_B"] = out["sen_id_B"].map(score_map)

    out["winner_A"] = [
        compare_pair(a, b, direction, tie_margin)
        for a, b in zip(out["score_A"], out["score_B"])
    ]
    return out

def compute_model_battle(llama_df, chatgpt_df, sampled_pairs, metric_name, metric_col, direction, tie_margin=0.3):
    # 只保留 cross-label pairs
    battle_pairs = sampled_pairs[sampled_pairs["label_A"] != sampled_pairs["label_B"]].copy()

    # gold winner: non-greenwashing (label=0) should win
    battle_pairs["gold_winner_A"] = (battle_pairs["label_A"] == NON_GREENWASH_LABEL).astype(int)

    llama_pairs = build_metric_pairwise_outcomes(
        df_scores=llama_df,
        sampled_pairs=battle_pairs,
        metric_col=metric_col,
        direction=direction,
        tie_margin=tie_margin
    )[["sen_id_A", "sen_id_B", "winner_A"]].rename(columns={"winner_A": "winner_A_llama"})

    chatgpt_pairs = build_metric_pairwise_outcomes(
        df_scores=chatgpt_df,
        sampled_pairs=battle_pairs,
        metric_col=metric_col,
        direction=direction,
        tie_margin=tie_margin
    )[["sen_id_A", "sen_id_B", "winner_A"]].rename(columns={"winner_A": "winner_A_chatgpt"})

    merged = battle_pairs.merge(llama_pairs, on=["sen_id_A", "sen_id_B"], how="left")
    merged = merged.merge(chatgpt_pairs, on=["sen_id_A", "sen_id_B"], how="left")

    # 只保留兩模型都有明確勝負的 pair
    merged = merged.dropna(subset=["winner_A_llama", "winner_A_chatgpt"]).copy()

    if len(merged) == 0:
        return {
            "metric": metric_name,
            "n_pairs_used": 0,
            "llama_accuracy": np.nan,
            "chatgpt_accuracy": np.nan,
            "chatgpt_win_rate": np.nan,
            "llama_win_rate": np.nan,
            "both_correct_rate": np.nan,
            "both_wrong_rate": np.nan
        }

    merged["llama_correct"] = (merged["winner_A_llama"].astype(int) == merged["gold_winner_A"].astype(int))
    merged["chatgpt_correct"] = (merged["winner_A_chatgpt"].astype(int) == merged["gold_winner_A"].astype(int))

    # battle outcome
    def decide_battle(row):
        if row["chatgpt_correct"] and not row["llama_correct"]:
            return "chatgpt_win"
        elif row["llama_correct"] and not row["chatgpt_correct"]:
            return "llama_win"
        elif row["chatgpt_correct"] and row["llama_correct"]:
            return "both_correct"
        else:
            return "both_wrong"

    merged["battle_result"] = merged.apply(decide_battle, axis=1)

    total = len(merged)

    return {
        "metric": metric_name,
        "n_pairs_used": total,
        "llama_accuracy": merged["llama_correct"].mean(),
        "chatgpt_accuracy": merged["chatgpt_correct"].mean(),
        "chatgpt_win_rate": (merged["battle_result"] == "chatgpt_win").mean(),
        "llama_win_rate": (merged["battle_result"] == "llama_win").mean(),
        "both_correct_rate": (merged["battle_result"] == "both_correct").mean(),
        "both_wrong_rate": (merged["battle_result"] == "both_wrong").mean()
    }

# ===== 讀檔 =====
llama_df = pd.read_csv("greenclaims_groundtruth_llama32_3b_hybrid_retrieval_fewshot_7metrics_sentence_scored.csv")
chatgpt_df = pd.read_csv("greenclaims_groundtruth_chatgpt_hybrid_retrieval_fewshot_7metrics_sentence_scored.csv")
sampled_pairs = pd.read_csv("bt_outputs_rewrite_2400/fixed_sampled_pairs.csv")

# ===== 跑 battle =====
rows = []
for metric_name, info in METRICS.items():
    rows.append(
        compute_model_battle(
            llama_df=llama_df,
            chatgpt_df=chatgpt_df,
            sampled_pairs=sampled_pairs,
            metric_name=metric_name,
            metric_col=info["col"],
            direction=info["direction"],
            tie_margin=PAIR_TIE_MARGIN
        )
    )

battle_df = pd.DataFrame(rows)
print(battle_df)

battle_df.to_csv(
    "bt_outputs_rewrite_2400/model_battle_llama_vs_chatgpt.csv",
    index=False,
    encoding="utf-8-sig"
)

          metric  n_pairs_used  llama_accuracy  chatgpt_accuracy  \
0    specificity           672        0.656250          0.650298   
1       evidence           386        0.694301          0.709845   
2      vagueness           675        0.619259          0.480000   
3     commitment           653        0.826953          0.799387   
4       temporal           291        0.838488          0.835052   
5     deflection           480        0.787500          0.577083   
6  comparability           370        0.618919          0.624324   

   chatgpt_win_rate  llama_win_rate  both_correct_rate  both_wrong_rate  
0          0.061012        0.066964           0.589286         0.282738  
1          0.046632        0.031088           0.663212         0.259067  
2          0.042963        0.182222           0.437037         0.337778  
3          0.035222        0.062787           0.764165         0.137825  
4          0.020619        0.024055           0.814433         0.140893  
5          